<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/aggregate-entity-group/notebooks/training-a%20new-tokenizer-from-old.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [2]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 'S',
 '##yl',
 '##va',
 '##in',
 'and',
 'I',
 'work',
 'at',
 'Hu',
 '##gging',
 'Face',
 'in',
 'Brooklyn',
 '.',
 '[SEP]']

In [3]:
encoding.word_ids()

[None, 0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]

In [4]:
encoding.sequence_ids()

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None]

In [5]:
start, end = encoding.word_to_chars(3)
example[end]

' '

In [6]:
from transformers import pipeline
token_classifier = pipeline("token-classification")
token_classifier("This person name is Sylvain and he works at Huggin Face in US.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity': 'I-PER',
  'score': np.float32(0.9995653),
  'index': 5,
  'word': 'S',
  'start': 20,
  'end': 21},
 {'entity': 'I-PER',
  'score': np.float32(0.9987286),
  'index': 6,
  'word': '##yl',
  'start': 21,
  'end': 23},
 {'entity': 'I-PER',
  'score': np.float32(0.9982584),
  'index': 7,
  'word': '##va',
  'start': 23,
  'end': 25},
 {'entity': 'I-PER',
  'score': np.float32(0.99941564),
  'index': 8,
  'word': '##in',
  'start': 25,
  'end': 27},
 {'entity': 'I-ORG',
  'score': np.float32(0.99618644),
  'index': 13,
  'word': 'Hu',
  'start': 44,
  'end': 46},
 {'entity': 'I-ORG',
  'score': np.float32(0.9213494),
  'index': 14,
  'word': '##gg',
  'start': 46,
  'end': 48},
 {'entity': 'I-ORG',
  'score': np.float32(0.986147),
  'index': 15,
  'word': '##in',
  'start': 48,
  'end': 50},
 {'entity': 'I-ORG',
  'score': np.float32(0.9868825),
  'index': 16,
  'word': 'Face',
  'start': 51,
  'end': 55},
 {'entity': 'I-LOC',
  'score': np.float32(0.9993198),
  'index': 18,
  

In [7]:
token_classifier = pipeline("token-classification", aggregation_strategy='simple')
token_classifier("This person name is Sylvain and he works at Hugging Face in US.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.99901664),
  'word': 'Sylvain',
  'start': 20,
  'end': 27},
 {'entity_group': 'ORG',
  'score': np.float32(0.99339485),
  'word': 'Hugging Face',
  'start': 44,
  'end': 56},
 {'entity_group': 'LOC',
  'score': np.float32(0.99935967),
  'word': 'US',
  'start': 60,
  'end': 62}]

In [8]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)
example = "This person name is Sylvain and he works at Hugging Face in US."
inputs = tokenizer(example, return_tensors='pt')
outputs = model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
print(inputs["input_ids"].shape)
print(outputs.logits.shape)

torch.Size([1, 20])
torch.Size([1, 20, 9])


In [10]:
import torch
probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)

[0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 0, 8, 0, 0]


In [11]:
model.config.id2label

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

In [12]:
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
  label = model.config.id2label[pred]
  if label != "0":
    results.append(
        {"entity": label, "score": probabilities[idx][pred], "word": tokens[idx]}
    )

print(results)

[{'entity': 'O', 'score': 0.9996699094772339, 'word': '[CLS]'}, {'entity': 'O', 'score': 0.9999431371688843, 'word': 'This'}, {'entity': 'O', 'score': 0.9999676942825317, 'word': 'person'}, {'entity': 'O', 'score': 0.999970555305481, 'word': 'name'}, {'entity': 'O', 'score': 0.9999674558639526, 'word': 'is'}, {'entity': 'I-PER', 'score': 0.9995612502098083, 'word': 'S'}, {'entity': 'I-PER', 'score': 0.998798131942749, 'word': '##yl'}, {'entity': 'I-PER', 'score': 0.9982945322990417, 'word': '##va'}, {'entity': 'I-PER', 'score': 0.9994126558303833, 'word': '##in'}, {'entity': 'O', 'score': 0.9999525547027588, 'word': 'and'}, {'entity': 'O', 'score': 0.9999501705169678, 'word': 'he'}, {'entity': 'O', 'score': 0.9999507665634155, 'word': 'works'}, {'entity': 'O', 'score': 0.9998213648796082, 'word': 'at'}, {'entity': 'I-ORG', 'score': 0.9973342418670654, 'word': 'Hu'}, {'entity': 'I-ORG', 'score': 0.9891602396965027, 'word': '##gging'}, {'entity': 'I-ORG', 'score': 0.9936902523040771, 'wo

In [13]:
results = []
input_with_offsets = tokenizer(example, return_offsets_mapping = True)
tokens = input_with_offsets.tokens()
offsets = input_with_offsets["offset_mapping"]

for idx, pred in enumerate(predictions):
  label = model.config.id2label[pred]
  if label != "0":
    start, end = offsets[idx]
    results.append(
        {
            "entity": label,
            "score": probabilities[idx][pred],
            "word": tokens[idx],
            "start": start,
            "end": end,
        }
    )
print(results)

[{'entity': 'O', 'score': 0.9996699094772339, 'word': '[CLS]', 'start': 0, 'end': 0}, {'entity': 'O', 'score': 0.9999431371688843, 'word': 'This', 'start': 0, 'end': 4}, {'entity': 'O', 'score': 0.9999676942825317, 'word': 'person', 'start': 5, 'end': 11}, {'entity': 'O', 'score': 0.999970555305481, 'word': 'name', 'start': 12, 'end': 16}, {'entity': 'O', 'score': 0.9999674558639526, 'word': 'is', 'start': 17, 'end': 19}, {'entity': 'I-PER', 'score': 0.9995612502098083, 'word': 'S', 'start': 20, 'end': 21}, {'entity': 'I-PER', 'score': 0.998798131942749, 'word': '##yl', 'start': 21, 'end': 23}, {'entity': 'I-PER', 'score': 0.9982945322990417, 'word': '##va', 'start': 23, 'end': 25}, {'entity': 'I-PER', 'score': 0.9994126558303833, 'word': '##in', 'start': 25, 'end': 27}, {'entity': 'O', 'score': 0.9999525547027588, 'word': 'and', 'start': 28, 'end': 31}, {'entity': 'O', 'score': 0.9999501705169678, 'word': 'he', 'start': 32, 'end': 34}, {'entity': 'O', 'score': 0.9999507665634155, 'wor

In [20]:
import numpy as np

results = []
input_with_offset = tokenizer(example, return_offsets_mapping=True)
tokens = input_with_offset.tokens()
offsets = input_with_offset["offset_mapping"]

idx = 0
while idx < len(predictions):
  pred = predictions[idx]
  label = model.config.id2label[pred]
  if label != 'O':
    label = label[2:]
    start, _ = offsets[idx]

    all_score = []
    while idx < len(predictions) and model.config.id2label[predictions[idx]] == f"I-{label}":
      all_score.append(probabilities[idx][predictions[idx]])
      _, end = offsets[idx]
      idx += 1

    # Only append the final aggregated entity group after processing all its sub-tokens
    score = np.mean(all_score).item()
    word = example[start:end]
    results.append(
        {
            "entity-group": label,
            "score": score,
            "word": word,
            "start": start,
            "end": end
        }
    )
  else:
    idx += 1

print(results)

[{'entity-group': 'PER', 'score': 0.9990166425704956, 'word': 'Sylvain', 'start': 20, 'end': 27}, {'entity-group': 'ORG', 'score': 0.9933949112892151, 'word': 'Hugging Face', 'start': 44, 'end': 56}, {'entity-group': 'LOC', 'score': 0.9993595480918884, 'word': 'US', 'start': 60, 'end': 62}]
